# Phase 3, experiment 2: is a simpler model more stable?

**Why this experiment:** In `03_degree_features.ipynb`, adding three degree features lowered the
validation AUC by 0.0376, but the public leaderboard went up by 0.0011. The validation drop came almost
entirely from four time steps (24, 29, 31, 32), and the model barely used the new features. The default
LightGBM also scores a perfect 1.0 on the training rows, which means it is overfitting. So the
validation score may be unstable rather than informative.

**What is tested:** Four models, each changing one thing from the default, and
each trained on two feature sets (raw, and raw + degree):

| Name | Setting | Idea |
|---|---|---|
| default | `num_leaves=31`, `learning_rate=0.1`, 100 trees | the reference from earlier notebooks |
| leaves_15 | `num_leaves=15` | mildly simpler trees |
| leaves_7 | `num_leaves=7` | much simpler trees |
| slow | `learning_rate=0.03`, 300 trees | many small steps instead of few large ones |

`num_leaves` is the maximum number of leaves in each tree. Fewer leaves means each tree can only learn
broad patterns. `learning_rate` is how much each new tree is allowed to correct the previous ones.

Same split as before: train on steps 1 to 21, validate on steps 22 to 35. No submission is made here.

## 1. Load and build the features 

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")

# Degree features, counted over all edges 
out_deg = edges["txId1"].value_counts()
in_deg = edges["txId2"].value_counts()

def add_degree_features(df):
    df = df.copy()
    df["in_degree"] = df["txId"].map(in_deg).fillna(0).astype(int)
    df["out_degree"] = df["txId"].map(out_deg).fillna(0).astype(int)
    df["total_degree"] = df["in_degree"] + df["out_degree"]
    return df

train = add_degree_features(train)

raw_cols = [c for c in train.columns if c.startswith("feat_")]
degree_cols = ["in_degree", "out_degree", "total_degree"]

feature_sets = {
    "raw": raw_cols,
    "raw+degree": raw_cols + degree_cols,
}

In [ ]:
# Labeled rows only, then the same time-based split as before
labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

TRAIN_LAST_STEP = 21
tr = labeled[labeled["time_step"] <= TRAIN_LAST_STEP]
va = labeled[labeled["time_step"] > TRAIN_LAST_STEP]
y_tr, y_va = tr["label"], va["label"]

WEAK_STEPS = [24, 29, 31, 32]
OTHER_STEPS = [s for s in sorted(va["time_step"].unique()) if s not in WEAK_STEPS]

print("Train rows:", len(tr), " Validation rows:", len(va))
print("Weak steps:", WEAK_STEPS)
print("Other steps:", OTHER_STEPS)

Train rows: 19530  Validation rows: 11705
Weak steps: [24, 29, 31, 32]
Other steps: [np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(30), np.int64(33), np.int64(34), np.int64(35)]


## 2. The four model settings

In [ ]:
model_settings = {
    "default": {},
    "leaves_15": {"num_leaves": 15},
    "leaves_7": {"num_leaves": 7},
    "slow": {"learning_rate": 0.03, "n_estimators": 300},
}

## 3. Train every combination and record the scores

For each model setting and each feature set (4 x 2 = 8 models), this records:

- the AUC on the training rows (1.0 means the model has memorised them)
- the AUC on all validation rows
- the AUC for each validation step separately

In [ ]:
summary_rows = []    # one row per model: overall scores
per_step_auc = {}    # (setting name, feature set name) -> AUC for each validation step

for setting_name, params in model_settings.items():
    for fs_name, cols in feature_sets.items():
        # **params passes the dictionary entries as keyword arguments
        model = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **params)
        model.fit(tr[cols], y_tr)

        tr_pred = model.predict_proba(tr[cols])[:, 1]
        va_pred = model.predict_proba(va[cols])[:, 1]

        # AUC for each validation step
        step_auc = {}
        for step in sorted(va["time_step"].unique()):
            in_step = (va["time_step"] == step).values   
            step_auc[step] = roc_auc_score(y_va[in_step], va_pred[in_step])
        step_auc = pd.Series(step_auc)
        per_step_auc[(setting_name, fs_name)] = step_auc

        summary_rows.append({
            "setting": setting_name,
            "features": fs_name,
            "train_auc": roc_auc_score(y_tr, tr_pred),
            "val_auc": roc_auc_score(y_va, va_pred),
            "mean_auc_weak_steps": step_auc[WEAK_STEPS].mean(),
            "mean_auc_other_steps": step_auc[OTHER_STEPS].mean(),
        })
        print("done:", setting_name, "/", fs_name)

summary = pd.DataFrame(summary_rows)

done: default / raw
done: default / raw+degree
done: leaves_15 / raw
done: leaves_15 / raw+degree
done: leaves_7 / raw
done: leaves_7 / raw+degree
done: slow / raw
done: slow / raw+degree


## 4. Results

### 4.1 Overall scores for the 8 models

`mean_auc_weak_steps` is the average of the per-step AUC over steps 24, 29, 31 and 32.
`mean_auc_other_steps` is the same average over the other ten validation steps.

In [5]:
summary.round(4)

,setting,features,train_auc,val_auc,mean_auc_weak_steps,mean_auc_other_steps
0,default,raw,1.0000,0.9148,0.7933,0.9866
1,default,raw+degree,1.0000,0.8772,0.7306,0.9846
2,leaves_15,raw,1.0000,0.8937,0.7613,0.9852
3,leaves_15,raw+degree,1.0000,0.8980,0.7644,0.9861
4,leaves_7,raw,0.9998,0.9058,0.7805,0.9851
5,leaves_7,raw+degree,0.9998,0.9067,0.7820,0.9855
6,slow,raw,1.0000,0.8929,0.7532,0.9851
7,slow,raw+degree,1.0000,0.8922,0.7553,0.9842


### 4.2 How much does adding degree features change validation, for each setting?

In notebook 03 the change was -0.0376 for the default model. If a simpler model is more stable, the
change should be much closer to zero.

In [6]:
# One row per setting, one column per feature set, then the difference between them
val_table = summary.pivot(index="setting", columns="features", values="val_auc")
val_table["change_from_degree"] = val_table["raw+degree"] - val_table["raw"]

# Keep the settings in the order they were defined
val_table = val_table.loc[list(model_settings.keys())]
val_table.round(4)

features,raw,raw+degree,change_from_degree
setting,,,
default,0.9148,0.8772,-0.0376
leaves_15,0.8937,0.8980,0.0043
leaves_7,0.9058,0.9067,0.0009
slow,0.8929,0.8922,-0.0007


### 4.3 AUC on the four weak steps, for every model

In [7]:
# Rows = weak steps, columns = (setting, feature set)
weak_table = pd.DataFrame(per_step_auc).loc[WEAK_STEPS]
weak_table.round(3)

default            leaves_15            leaves_7              slow  \
       raw raw+degree       raw raw+degree      raw raw+degree    raw   
24   0.787      0.743     0.759      0.769    0.775      0.769  0.752   
29   0.756      0.694     0.728      0.740    0.762      0.750  0.724   
31   0.789      0.719     0.765      0.752    0.776      0.790  0.747   
32   0.841      0.766     0.793      0.796    0.809      0.819  0.789   

               
   raw+degree  
24      0.749  
29      0.724  
31      0.753  
32      0.796

## Findings

- **Does a simpler model reduce overfitting (train AUC)?** No. Train AUC is 1.0 for the default,
  `leaves_15` and `slow` models, and 0.9998 for `leaves_7`. None of these settings is simple enough to
  stop the model fitting the training rows almost perfectly.
- **Does a simpler model score better or worse on validation?** Slightly worse on raw features. The
  default scores 0.9148, against 0.9058 (`leaves_7`), 0.8937 (`leaves_15`) and 0.8929 (`slow`).
- **Is the gap between "raw" and "raw + degree" smaller for the simpler models?** Yes, much smaller.
  Adding degree features changes validation by -0.0376 for the default model, but only by +0.0043
  (`leaves_15`), +0.0009 (`leaves_7`) and -0.0007 (`slow`). So the large drop seen in notebook 03 was
  specific to the default settings and not a consistent effect of the degree features.
- **How noisy is the validation score?** The eight models are all similar, yet their validation AUC
  ranges from 0.877 to 0.915. Differences of about 0.01 to 0.02 on this split cannot be told apart
  from noise with a single run.
- **What happens on the weak steps?** They stay weak for every model. The average AUC over steps 24,
  29, 31 and 32 is between 0.73 and 0.79 for all eight models, while the average over the other ten
  steps is about 0.985 in every case. Almost all the variation between models comes from the weak steps.
  This means the weak steps are a real problem with these time periods, not only an effect of overfitting.

